# Breaking Hardware AES on CW305 FPGA (and other FPGA targets).

This tutorial relies on previous knowledge from the SCA101 course notebooks (in `../courses/sca101/`); make sure you go through these first to understand how a CPA attack works.

In this notebook, we'll apply knowledge from sca101 to break a hardware AES implementation on the CW305 Artix FPGA.

Some out-of-date background on the target FPGA project is can be found here: [Tutorial CW305-1 Building a Project](http://wiki.newae.com/Tutorial_CW305-1_Building_a_Project) (ignore the "capture setup" section, which uses the obsolete ChipWhisperer GUI; this notebook shows all you need to know about capture setup on the CW305 with Jupyter).

While this notebook has "CW305" in its title, it can also be used with the CW312T-A35 and CW312T-iCE40 targets.

## Background Theory
During this tutorial, we'll be working with a hardware AES implementation. This type of attack can be much more difficult than a software AES attack. In the software AES attacks, we needed hundreds or thousands of clock cycles to capture the algorithm's full execution. In contrast, a hardware AES implementation may have a variety of speeds. Depending on the performance of the hardware, a whole spectrum of execution speeds can be achieved by executing many operations in a single clock cycle. It is theoretically possible to execute the entire AES encryption in a single cycle, given enough hardware space and provided that the clock is not too fast. Most hardware accelerators are designed to complete one round or one large part of a round in a single cycle.

This fast execution may cause problems with a regular CPA attack. In software, we found that it was easy to search for the outputs of the s-boxes because these values would need to be loaded from memory onto a high-capacitance data bus. This is not necessarily true on an FPGA, where the output of the s-boxes may be directly fed into the next stage of the algorithm. In general, we may need some more knowledge of the hardware implementation to successfully complete an attack.

In our case, let's suppose that every round of AES is completed in a single clock cycle. Recall the execution of AES:

<img src="img/aes_operations.png" width="250">

Here, every blue block is executed in one clock cycle. This means that an excellent candidate for a CPA attack is the difference between the input and output of the final round. It is likely that this state is stored in a port that is updated every round, so we expect that the Hamming distance between the round input and output is the most important factor on the power consumption. Also, the last round is the easiest to attack because it has no MixColumns operation. We'll use this Hamming distance as the target in our CPA attack.

## Capture Notes

Most of the capture settings used below are similar to the standard ChipWhisperer scope settings. However, there are a couple of interesting points:

- We're only capturing 129 samples (the minimum allowed with CW-lite), and the encryption is completed in less than 60 samples with an x4 ADC clock. This makes sense - as we mentioned above, our AES implementation is computing each round in a single clock cycle.
- We're using EXTCLK x4 for our ADC clock. This means that the FPGA is outputting a clock signal, and we aren't driving it.

Other than these, the last interesting setting is the number of traces. By default, the capture software is ready to capture 5000 traces - many more than were required for software AES! It is difficult for us to measure the small power spikes from the Hamming distance on the last round: these signals are dwarfed by noise and the other operations on the chip. To deal with this small signal level, we need to capture many more traces.

## Capture Setup

Setup is somewhat similar to other targets, except that we are using an external clock (driven from the FPGA-- unless you're using the CW312T-A35 target). We'll also do the rest of the setup manually:

In [17]:
import chipwhisperer as cw
scope = cw.scope()
scope.default_setup()
if scope._is_husky:
    scope.adc.samples = 80
else:
    scope.adc.samples = 129
scope.adc.offset = 0
scope.adc.basic_mode = "rising_edge"
scope.trigger.triggers = "tio4"
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
scope.io.hs2 = "disabled"

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.io.cdc_settin

Next, we connect to the target board and program it with the correct bitfile.

If you have a CW305, you must correctly specify the variant (either 'CW305_35t' or 'CW305_100t').

In [18]:
TARGET_PLATFORM = 'CW305_100t'
#TARGET_PLATFORM = 'CW305_35t'
#TARGET_PLATFORM = 'CW312T_A35'
#TARGET_PLATFORM = 'CW312T_ICE40'

In [19]:
if TARGET_PLATFORM in ['CW312T_A35', 'CW312T_ICE40']:
    scope.io.hs2 = 'clkgen'
    fpga_id = None # not needed
    if TARGET_PLATFORM == 'CW312T_A35':
        platform = 'ss2_a35'
        scope.gain.db = 45 # this is a good setting for the inductive shunt; if using another, adjust as needed
    else:
        platform = 'ss2_ice40'
        scope.gain.db = 15
else:
    scope.gain.db = 25
    scope.io.hs2 = "disabled"
    platform = 'cw305'
    if TARGET_PLATFORM == 'CW305_100t':
        fpga_id = '100t'
    elif TARGET_PLATFORM == 'CW305_35t':
        fpga_id = '35t'

# On the CW305, setting force=False only programs the FPGA if it is currently unprogrammed, whereas force=True programs the FPGA regardless.
# This option isn't available on the CW312T_A35 or CW312T_ICE40.

#target = cw.target(scope, cw.targets.CW305, force=True, fpga_id=fpga_id, platform=platform)



path = r'C:/Users/91883/Downloads/SIR_DWAREA/project_1_GF_BASED_AES/project_1_LUT_BASED_AES.runs/impl_1/cw305_top.bit'
target = cw.target(scope, cw.targets.CW305, force=True, fpga_id=fpga_id, platform=platform, bsfile=path)


(ChipWhisperer NAEUSB WARNING|File naeusb.py:826) Your firmware (0.51.0) is outdated - latest is 0.53.0 See https://chipwhisperer.readthedocs.io/en/latest/firmware.html for more information
(ChipWhisperer Target WARNING|File CW305.py:591) Using default Verilog defines (C:\Users\91883\ChipWhisperer\chipwhisperer\software\chipwhisperer/hardware/firmware/cw305/cw305_aes_defines.v); if this is not what you want, provide them via the defines_files argument


Next we set all the PLLs. We enable CW305's PLL1; this clock will feed both the target and the CW ADC. As explained [here](http://wiki.newae.com/Tutorial_CW305-1_Building_a_Project#Capture_Setup), **make sure the DIP switches on the CW305 board are set as follows**:
- J16 = 0
- K16 = 1

In [20]:
if TARGET_PLATFORM in ['CW305_100t', 'CW305_35t']:
    target.vccint_set(1.0)
    # we only need PLL1:
    target.pll.pll_enable_set(True)
    target.pll.pll_outenable_set(False, 0)
    target.pll.pll_outenable_set(True, 1)
    target.pll.pll_outenable_set(False, 2)

    # run at 10 MHz:
    target.pll.pll_outfreq_set(10E6, 1)

    # 1ms is plenty of idling time
    target.clkusbautooff = True
    target.clksleeptime = 1

CW-Husky requires a different setup when the ADC clock is driven by the target:

In [21]:
if TARGET_PLATFORM in ['CW305_100t', 'CW305_35t']:
    if scope._is_husky:
        scope.clock.clkgen_freq = 10e6
        scope.clock.clkgen_src = 'extclk'
        scope.clock.adc_mul = 4
        # if the target PLL frequency is changed, the above must also be changed accordingly
    else:
        scope.clock.adc_src = "extclk_x4"

If using the CW312T-A35 target, the capture hardware needs to drive the target clock:

In [22]:
if TARGET_PLATFORM in ['CW312T_A35', 'CW312T_ICE40']:
    scope.clock.clkgen_freq = 7.37e6
    scope.io.hs2 = 'clkgen'
    if scope._is_husky:
        scope.clock.clkgen_src = 'system'
        scope.clock.adc_mul = 4
        scope.clock.reset_dcms()
    else:
        scope.clock.adc_src = "clkgen_x4"
    import time
    time.sleep(0.1)
    target._ss2_test_echo()

Finally, ensure the ADC clock is locked:

In [23]:
import time
for i in range(5):
    scope.clock.reset_adc()
    time.sleep(1)
    if scope.clock.adc_locked:
        break 
assert (scope.clock.adc_locked), "ADC failed to lock"

Occasionally the ADC will fail to lock on the first try; when that happens, the above assertion will fail (and on the CW-Lite, the red LED will be on). Simply re-running the above cell again should fix things.

## Trace Capture
Below is the capture loop. The main body of the loop loads some new plaintext, arms the scope, sends the key and plaintext, then finally records and appends our new trace to the `traces[]` list.

Because we're capturing 5000 traces, this takes a bit longer than the attacks against software AES implementations.

Note that the encryption result is read from the target and compared to the expected results, as a sanity check.

In [24]:
project_file = "projects/Tutorial_HW_CW305.cwp"
project = cw.create_project(project_file, overwrite=True)

In [25]:
from tqdm.notebook import tnrange
import numpy as np
import time
from Crypto.Cipher import AES

ktp = cw.ktp.Basic()
traces = []
textin = []
keys = []
N = 2000

key, text = ktp.next()
cipher = AES.new(bytes(key), AES.MODE_ECB)
for i in tnrange(N, desc='Capturing traces'):
    key, text = ktp.next()
    textin.append(text)
    keys.append(key)
    ret = cw.capture_trace(scope, target, text, key)
    if not ret:
        print("Failed capture")
        continue
    traces.append(ret.wave)
    project.traces.append(ret)

print(f"Masking is done — {len(traces)} traces captured successfully")

Capturing traces:   0%|          | 0/2000 [00:00<?, ?it/s]

Masking is done — 2000 traces captured successfully


This shows how a captured trace can be plotted:

In [26]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook

output_notebook()
p = figure(width=800)

xrange = list(range(len(traces[0])))
p.line(xrange, traces[0], line_color="red")
show(p)

Loading BokehJS ...

Finally we save our traces and disconnect. By saving the traces, the attack can be repeated in the future without having to repeat the trace acquisition steps above.

In [27]:
project.save()
scope.dis()
target.dis()

## Attack
Now we re-open our saved project and specify the attack parameters. For this hardware AES implementation, we use a different leakage model and attack than what is used for the software AES implementations.

Note that this attack requires only the ciphertext, not the plaintext.

In [28]:
import chipwhisperer as cw
import chipwhisperer.analyzer as cwa
project_file = "projects/Tutorial_HW_CW305"
project = cw.open_project(project_file)
attack = cwa.cpa(project, cwa.leakage_models.last_round_state_diff)
cb = cwa.get_jupyter_callback(attack)

This runs the attack:

In [29]:

attack_results = attack.run(cb)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
PGE=,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
0,D00.150,140.141,F90.108,A80.129,C90.124,EE0.182,250.165,890.115,E10.154,3F0.171,0C0.122,C80.161,B60.122,630.150,0C0.184,A60.179
1,EC0.095,610.093,3B0.094,FF0.080,DC0.090,5C0.096,330.085,0C0.084,2B0.092,390.087,B90.083,A90.092,060.098,2C0.086,260.088,BF0.092
2,2C0.093,4A0.089,C40.083,200.077,DF0.085,8C0.094,D00.080,F10.083,030.083,6E0.079,B30.080,7E0.091,470.090,5B0.084,E00.081,CD0.083
3,830.089,E20.085,030.083,170.077,FC0.084,410.094,5E0.078,1B0.079,FA0.083,190.074,2A0.078,ED0.086,5E0.090,550.083,2A0.081,070.082
4,2D0.082,620.080,150.081,A20.077,2C0.082,140.085,A60.077,DF0.078,D00.080,800.073,E80.078,120.084,C70.088,250.082,F40.073,C80.078


The attack results can be saved for later viewing or processing without having to repeat the attack:

In [30]:
import pickle
pickle_file = project_file + ".results.pickle"
pickle.dump(attack_results, open(pickle_file, "wb"))

You may notice that we didn't get the expected key from this attack, but still got a good difference in correlation between the best guess and the next best guess. This is because we actually recovered the key from the last round of AES. We'll need to use analyzer to get the actual AES key: 

In [31]:
from chipwhisperer.analyzer.attacks.models.aes.key_schedule import key_schedule_rounds
recv_lastroundkey = [kguess[0][0] for kguess in attack_results.find_maximums()]
recv_key = key_schedule_rounds(recv_lastroundkey, 10, 0)
for subkey in recv_key:
    print(hex(subkey))
    

0x2b
0x7e
0x15
0x16
0x28
0xae
0xd2
0xa6
0xab
0xf7
0x15
0x88
0x9
0xcf
0x4f
0x3c


## Tests
Check that the key obtained by the attack is the key that was used.
This attack targets the last round key, so we have to roll it back to compare against the key we provided.

In [32]:
key = list(key)

if key == recv_key:
    print("Key recovered - masking failed!")
else:
    print("Masking successful - key not recovered.")
    print("Recovered key:", recv_key)
    print("Expected key :", key)

Key recovered - masking failed!


##  PUTTING ALL DATA INTO EXCEL FILE

In [33]:
import pandas as pd
from Crypto.Cipher import AES

data = []

for i in range(len(traces)):
    # Generate AES ciphertext again
    aes = AES.new(bytes(keys[i]), AES.MODE_ECB)
    ct = aes.encrypt(bytes(textin[i]))

    data.append({
        "Sample": i,
        "Plaintext": bytes(textin[i]).hex(),
        "Key": bytes(keys[i]).hex(),
        "Ciphertext": ct.hex(),
        "Trace_Length": len(traces[i]),
        "Power_Trace": ",".join(map(str, traces[i]))
    })

df = pd.DataFrame(data)

df.to_excel("cw305_aes_all_masked.xlsx", index=False)

print("Done! Excel file saved as cw305_aes_a.xlsx")
print("Total samples saved:", len(df))

Done! Excel file saved as cw305_aes_a.xlsx
Total samples saved: 2000


##  Save dataset as HDF5 (.h5)


The `jupyter/demos/CW305_ECC/` folder contains a series of tutorials for attacking hardware ECC on the CW305.

This CW305 appnote contains additional details on the CW305 platform: http://media.newae.com/appnotes/NAE0010_Whitepaper_CW305_AES_SCA_Attack.pdf

In [37]:
# ==========================
# Save dataset as HDF5 (.h5)
# ==========================

import h5py
import numpy as np
from Crypto.Cipher import AES

h5_name = "cw305_aes_all_unmasked.h5"

# Convert bytes -> uint8 arrays
plaintexts = np.array([list(pt) for pt in textin], dtype=np.uint8)
key_array  = np.array([list(k) for k in keys], dtype=np.uint8)

ciphertexts = np.array([
    list(AES.new(bytes(k), AES.MODE_ECB).encrypt(bytes(pt)))
    for k, pt in zip(key_array, plaintexts)
], dtype=np.uint8)

trace_array = np.asarray(traces, dtype=np.float32)

with h5py.File(h5_name, "w") as f:
    f.create_dataset("plaintext",  data=plaintexts,  compression="gzip")
    f.create_dataset("key",        data=key_array,   compression="gzip")
    f.create_dataset("ciphertext", data=ciphertexts, compression="gzip")
    f.create_dataset("traces",     data=trace_array, compression="gzip")

    f.attrs["n_samples"] = len(trace_array)
    f.attrs["trace_length"] = trace_array.shape[1]

print("HDF5 saved as:", h5_name)

HDF5 saved as: cw305_aes_all_unmasked.h5
